This notebook creates dummy source data and configuration files to simulate incoming data in a landing zone. It's essentially a setup script for testing the ingestion process.

In [0]:
spark.version

'4.1.0'

In [0]:
from datetime import datetime

# Create a widget for manual seed control (optional)
dbutils.widgets.text(
    name="random_seed",
    defaultValue="",
    label="Random Seed (leave empty to use current hour)"
)

# Get the widget value
widget_seed = dbutils.widgets.get("random_seed").strip()

# Decide final seed
if widget_seed and widget_seed.isdigit():
    seed_value = int(widget_seed)
    print(f"Using explicit seed from widget: {seed_value}")
else:
    # Fallback: use current hour (0–23) — changes every hour
    seed_value = datetime.now().hour
    print(f"No valid seed provided → using current hour as seed: {seed_value}")

# Create a widget for manual seed control (optional)
dbutils.widgets.text(
    name="catalog_name",
    defaultValue="",
    label="catalog name"
)



Using explicit seed from widget: 22


In [0]:
catalog = dbutils.widgets.get("catalog_name").strip()

print(f"catalog: {catalog}")

catalog: akhtar


In [0]:
# --- UC managed storage root (from your error message) ---
uc_root = "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727"

# --- Project paths under UC root ---
project_root = f"{uc_root}/macro_platform"



landing_base = f"{project_root}/landing/lse/equities_prices_test/"
config_base  = f"{project_root}/config/"
print("Landing base:", landing_base)
print("Config base :", config_base)

# Create folders (idempotent)
dbutils.fs.mkdirs(landing_base)
dbutils.fs.mkdirs(config_base)

# Optional: clean the landing folder to re-run tests from scratch
# dbutils.fs.rm(landing_base, recurse=True)
# dbutils.fs.mkdirs(landing_base)

display(dbutils.fs.ls(f"{project_root}/landing/lse/"))

Landing base: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/
Config base : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/config/


path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/,equities_prices_test/,0,1769788581000


In [0]:
landing_path: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/"
raw_checkpoint: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/"
schema_location: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/"
raw_table: "raw.lse_equities_prices_test"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

currencies = ["GBP", "USD", "EUR", "JPY"]
# Base "good" dataset
n = 1200  # above expected_min_rows=1000
df_good = (
    spark.range(0, n)
    .withColumn("isin", F.concat(F.element_at(F.array([F.lit(c) for c in currencies]), (F.rand(seed=seed_value) * len(currencies)).cast("int") + 1), F.lit("00TEST"), F.lpad(F.col("id").cast("string"), 6, "0")))
    .withColumn("trade_date", F.date_sub(F.current_date(), (F.col("id") % 30).cast("int")))
    .withColumn("close_price", (F.rand(seed=seed_value) * 1000).cast("decimal(18,6)"))
    .withColumn("currency", F.element_at(F.array([F.lit(c) for c in currencies]), (F.rand(seed=seed_value) * len(currencies)).cast("int") + 1))
    .select("isin", "trade_date", "close_price", "currency")
)

# Case 1: too few rows -> should trigger missing_rows check (STOP)
df_few = df_good.limit(50)

# Case 2: required columns contain nulls -> should trigger required_nulls check (STOP)
df_required_nulls = (
    df_good
    .withColumn("isin", F.when(F.rand(seed=seed_value) < 0.05, F.lit(None)).otherwise(F.col("isin")))
    .withColumn("trade_date", F.when(F.rand(seed=seed_value) < 0.02, F.lit(None).cast("date")).otherwise(F.col("trade_date")))
)

# Case 3: high null rate on optional column close_price -> should trigger WARN
df_high_null_rate = (
    df_good
    .withColumn("close_price",
                F.when(F.rand(seed=seed_value) < 0.50, F.lit(None).cast("decimal(18,6)"))
                 .otherwise(F.col("close_price")))
)


In [0]:

display(df_good.limit(50))

isin,trade_date,close_price,currency
EUR00TEST000000,2026-01-30,521.252590,EUR
USD00TEST000001,2026-01-29,475.777665,USD
GBP00TEST000002,2026-01-28,141.385320,GBP
JPY00TEST000003,2026-01-27,851.376216,JPY
JPY00TEST000004,2026-01-26,985.638729,JPY
JPY00TEST000005,2026-01-25,881.842793,JPY
USD00TEST000006,2026-01-24,387.988322,USD
USD00TEST000007,2026-01-23,323.431889,USD
USD00TEST000008,2026-01-22,456.117285,USD
EUR00TEST000009,2026-01-21,569.847518,EUR


In [0]:

# display(df_high_null_rate.limit(5))
df_high_null_rate.columns


['isin', 'trade_date', 'close_price', 'currency']

In [0]:

display(df_required_nulls.limit(5))

isin,trade_date,close_price,currency
EUR00TEST000000,2026-01-30,521.252590,EUR
USD00TEST000001,2026-01-29,475.777665,USD
GBP00TEST000002,2026-01-28,141.385320,GBP
JPY00TEST000003,2026-01-27,851.376216,JPY
JPY00TEST000004,2026-01-26,985.638729,JPY


In [0]:
from datetime import datetime, timezone

def write_timestamped_csv(df, base_dir, prefix, run_tag=None):
    """
    Writes df as ONE CSV file with a timestamped filename into base_dir.

    Example output:
      base_dir/good/good_20260130_0915.csv

    Spark writes to a folder; we write to a temp folder then move the single part file.
    """
    run_tag = run_tag or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
    target_dir = f"{base_dir.rstrip('/')}/{prefix}/"
    tmp_dir    = f"{target_dir}_tmp_{run_tag}/"
    out_file   = f"{target_dir}{prefix}_{run_tag}.csv"

    dbutils.fs.mkdirs(target_dir)

    # 1) write to temp folder (single part file)
    (df.coalesce(1)
      .write.mode("overwrite")
      .option("header", "true")
      .csv(tmp_dir))

    # 2) find the part file and move/rename to timestamped filename
    part_files = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.startswith("part-") and f.name.endswith(".csv")]
    if not part_files:
        raise RuntimeError(f"No part file found in {tmp_dir}. Contents: {[f.name for f in dbutils.fs.ls(tmp_dir)]}")

    dbutils.fs.mv(part_files[0], out_file, True)

    # 3) cleanup temp folder
    dbutils.fs.rm(tmp_dir, recurse=True)

    print(f"✅ Wrote {out_file}")
    return out_file

In [0]:
def write_csv_folder(df, path):
    (df.coalesce(1)
      .write.mode("overwrite")
      .option("header", "true")
      .csv(path))

In [0]:
run_tag = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

write_timestamped_csv(df_good,           landing_base, "good", run_tag)
write_timestamped_csv(df_few,            landing_base, "few_rows", run_tag)
write_timestamped_csv(df_required_nulls, landing_base, "required_nulls", run_tag)
write_timestamped_csv(df_high_null_rate, landing_base, "high_null_close", run_tag)

display(dbutils.fs.ls(landing_base))

✅ Wrote abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/good/good_20260130_1558.csv
✅ Wrote abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/few_rows_20260130_1558.csv
✅ Wrote abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/required_nulls/required_nulls_20260130_1558.csv
✅ Wrote abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/high_null_close/high_null_close_20260130_1558.csv


path,name,size,modificationTime
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/few_rows/,few_rows/,0,1769788706000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/good/,good/,0,1769788705000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/high_null_close/,high_null_close/,0,1769788708000
abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/required_nulls/,required_nulls/,0,1769788707000


In [0]:
yaml_text = f"""
pipeline_name: "macro-platform"
source_name: "LSE"
dataset_name: "equities_prices_test"
format: "csv"

landing_path: "{landing_base}"
raw_table: "raw.lse_equities_prices_test"

# Put schema + checkpoint under UC root too (so you don't hit permission errors)
raw_checkpoint: "{project_root}/checkpoints/raw/lse/equities_prices_test/"
schema_location: "{project_root}/schema/raw/lse/equities_prices_test/"

read_options:
  header: "true"
  delimiter: ","
  inferSchema: "false"

schema:
  - {{name: isin, type: string, nullable: false}}
  - {{name: trade_date, type: date, nullable: false}}
  - {{name: close_price, type: decimal(18,6), nullable: true}}
  - {{name: currency, type: string, nullable: true}}

dq:
  stop_on_severity: "ERROR"
  expected_min_rows: 1000
  required_columns: ["isin", "trade_date"]
  null_thresholds:
    close_price: 0.20

  policies:
    missing_rows:
      severity: "ERROR"
    required_nulls:
      severity: "ERROR"
    null_rate_threshold:
      severity: "WARN"
"""

cfg_path = config_base + "lse_prices_test.yml"
dbutils.fs.put(cfg_path, yaml_text, overwrite=True)

print("✅ Wrote config to:", cfg_path)
print(yaml_text)

Wrote 1295 bytes.
✅ Wrote config to: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/config/lse_prices_test.yml

pipeline_name: "macro-platform"
source_name: "LSE"
dataset_name: "equities_prices_test"
format: "csv"

landing_path: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/"
raw_table: "raw.lse_equities_prices_test"

# Put schema + checkpoint under UC root too (so you don't hit permission errors)
raw_checkpoint: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/"
schema_location: "abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/"

read_options:
  header: "true"
  delimiter: ","
  inferSchema: "false"

schema:
  - {name: isin, type: string, nullab

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS raw")
spark.sql("CREATE DATABASE IF NOT EXISTS curated")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("CREATE DATABASE IF NOT EXISTS governance")
print("✅ Databases ensured: raw, curated, gold, governance")

✅ Databases ensured: raw, curated, gold, governance
